In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!nvidia-smi

Fri Oct 17 09:53:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Load the Master Dataset ---
# This is the single, all-inclusive file you created.
final_file_path = '/content/drive/My Drive/ML_Price_Prediction/dataset/train_fully_featured.csv'
df_train = pd.read_csv(final_file_path)

print("✅ Master dataset loaded successfully.")
print(f"Shape: {df_train.shape}")


# --- 2. Prepare X (Features) and y (Target) ---
# We use log-price for better model performance.
df_train['log_price'] = np.log1p(df_train['price'])
y = df_train['log_price']

# Drop non-feature columns
features_to_drop = [
    'sample_id', 'catalog_content', 'image_link', 'price', 'log_price',
    'item_name', 'unit'
]
X = df_train.drop(columns=features_to_drop)
print(f"Feature matrix X created with shape: {X.shape}")


# --- 3. Define Feature Groups for Specialist Models ---
structured_features = [
    'brand', 'num_of_packs', 'net_qty', 'total_qty', 'is_qty_missing'
]
text_features = [col for col in X.columns if col.startswith('text_embed_')]
image_features = [col for col in X.columns if col.startswith('img_embed_')]

print(f"\nDefined {len(structured_features)} structured, {len(text_features)} text, and {len(image_features)} image features.")

✅ Master dataset loaded successfully.
Shape: (75000, 971)
Feature matrix X created with shape: (75000, 965)

Defined 5 structured, 384 text, and 576 image features.


new implementation


In [ ]:
# ====================================================================
# 🔹 MODELING - BLOCK 1: SETUP AND DATA LOADING
# ====================================================================
import pandas as pd
import numpy as np
import os

# --- Step 1.1: Load the final, cleaned datasets ---
BASE_PATH = '/content/drive/MyDrive/ML_Price_Prediction/dataset/'
# These are the new, high-quality files you just created
TRAIN_CLEAN_PATH = os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv')
TEST_CLEAN_PATH = os.path.join(BASE_PATH, 'test_final_cleaned_v2.csv')

df_train = pd.read_csv(TRAIN_CLEAN_PATH)
df_test = pd.read_csv(TEST_CLEAN_PATH)
print(f"✅ Successfully loaded new, cleaned datasets.")
print(f"Training data shape: {df_train.shape}")
print(f"Test data shape: {df_test.shape}")

# --- Step 1.2: Prepare X (Features) and y (Target) ---
df_train['log_price'] = np.log1p(df_train['price'])
y = df_train['log_price']

# Define the features to be used
features_to_drop = ['sample_id', 'catalog_content', 'image_link', 'price', 'log_price', 'item_name', 'unit']
features = [col for col in df_train.columns if col not in features_to_drop]

X = df_train[features]
X_test = df_test[features]
print(f"\nFeature matrix X created with shape: {X.shape}")
print(f"Feature matrix X_test created with shape: {X_test.shape}")

✅ Successfully loaded new, cleaned datasets.
Training data shape: (75000, 971)
Test data shape: (75000, 970)

Feature matrix X created with shape: (75000, 965)
Feature matrix X_test created with shape: (75000, 965)


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import os
import joblib

print("🚀 Starting Final, Restart-Proof Training, Prediction, AND Evaluation...")

# =============================
# 🔹 Step 1: Load Datasets and Define Paths
# =============================
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
PREDS_PATH = os.path.join(MODELS_PATH, 'fold_predictions/')
os.makedirs(PREDS_PATH, exist_ok=True)

df_train = pd.read_csv(os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv'))
df_test = pd.read_csv(os.path.join(BASE_PATH, 'test_final_cleaned_v2.csv'))

# Prepare X and y
df_train['log_price'] = np.log1p(df_train['price'])
y = df_train['log_price']
features_to_drop = ['sample_id', 'catalog_content', 'image_link', 'price', 'log_price', 'item_name', 'unit']
features = [col for col in df_train.columns if col not in features_to_drop]
X = df_train[features]
X_test = df_test[features]
print(f"Loaded data. X shape: {X.shape}, X_test shape: {X_test.shape}")


# =============================
# 🔹 Step 2: Restart-Proof Cross-Validation Training
# =============================
NFOLDS = 5
df_train['price_bins'] = pd.cut(df_train['price'], bins=10, labels=False)
skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=42)

# --- NEW: Initialize an array for Out-of-Fold (OOF) predictions ---
oof_preds = np.zeros(len(df_train))

lgb_params = {
    'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 5000,
    'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'num_leaves': 60, 'verbose': -1, 'n_jobs': -1, 'seed': 42
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, df_train['price_bins'])):

    test_pred_path = os.path.join(PREDS_PATH, f'test_preds_fold_{fold}.npy')
    oof_pred_path = os.path.join(PREDS_PATH, f'oof_preds_fold_{fold}.npy')

    # --- Checkpoint Logic: Checks if BOTH files for the fold exist ---
    if os.path.exists(test_pred_path) and os.path.exists(oof_pred_path):
        print(f"\n⏩ Skipping completed Fold {fold+1}...")
        # Load the saved OOF preds for this fold into the main array
        oof_preds[val_idx] = np.load(oof_pred_path)
        continue

    print(f"\n===== Running Fold {fold+1} =====")
    X_train, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

    # --- Safe Target Encoding ---
    target_mean = y_train_fold.groupby(X_train['brand']).mean()
    X_train['brand_encoded'] = X_train['brand'].map(target_mean).fillna(y_train_fold.mean())
    X_val['brand_encoded'] = X_val['brand'].map(target_mean).fillna(y_train_fold.mean())
    X_test_fold = X_test.copy()
    X_test_fold['brand_encoded'] = X_test_fold['brand'].map(target_mean).fillna(y_train_fold.mean())

    X_train_fold = X_train.drop(columns=['brand'])
    X_val_fold = X_val.drop(columns=['brand'])
    X_test_fold_final = X_test_fold.drop(columns=['brand'])

    # --- Train Model ---
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_train_fold, y_train_fold,
              eval_set=[(X_val_fold, y_val_fold)],
              eval_metric='mae', callbacks=[lgb.early_stopping(150, verbose=False)])

    # --- Predict on BOTH validation and test sets ---
    val_preds = model.predict(X_val_fold)
    fold_test_preds = model.predict(X_test_fold_final)

    # --- SAVE BOTH CHECKPOINTS ---
    oof_preds[val_idx] = val_preds
    np.save(oof_pred_path, val_preds) # Save OOF fold preds
    np.save(test_pred_path, fold_test_preds) # Save Test fold preds
    print(f"💾 Checkpoints for OOF and Test saved for Fold {fold+1}!")

# =============================
# 🔹 Step 3: Assemble Submission File
# =============================
print("\n\n✅ Training complete! Assembling final submission file...")
all_fold_preds = []
for fold in range(NFOLDS):
    all_fold_preds.append(np.load(os.path.join(PREDS_PATH, f'test_preds_fold_{fold}.npy')))
test_preds = np.mean(all_fold_preds, axis=0)
final_predictions = np.expm1(test_preds)
final_predictions[final_predictions < 0] = 0

submission_df = pd.DataFrame({'sample_id': df_test['sample_id'], 'price': final_predictions})
SUBMISSION_PATH = '/content/drive/My Drive/ML_Price_Prediction/submission_v2.csv'
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"🎉 Success! New submission file created at:\n{SUBMISSION_PATH}")

# =============================
# 🔹 Step 4: Create and Evaluate the NEW OOF "Report Card"
# =============================
print("\n💾 Creating new, accurate OOF 'report card' file...")
oof_df = pd.DataFrame({
    'sample_id': df_train['sample_id'],
    'price': df_train['price'],
    'predicted_price': np.expm1(oof_preds)
})
OOF_V2_PATH = os.path.join(BASE_PATH, 'oof_predictions_v2.csv')
oof_df.to_csv(OOF_V2_PATH, index=False)
print(f"✅ New report card saved to:\n{OOF_V2_PATH}")

# --- Final SMAPE Calculation ---
def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.nan_to_num(np.mean(numerator / denominator)) * 100

overall_smape = smape(oof_df['price'], oof_df['predicted_price'])
print("\n\n==========================================================")
print(f"🏆 Your NEW model's validation SMAPE score is: {overall_smape:.4f}%")
print("==========================================================")

🚀 Starting Final, Restart-Proof Training, Prediction, AND Evaluation...
Loaded data. X shape: (75000, 965), X_test shape: (75000, 965)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



===== Running Fold 1 =====


/tmp/ipython-input-2876328625.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['brand_encoded'] = X_train['brand'].map(target_mean).fillna(y_train_fold.mean())
/tmp/ipython-input-2876328625.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val['brand_encoded'] = X_val['brand'].map(target_mean).fillna(y_train_fold.mean())


💾 Checkpoints for OOF and Test saved for Fold 1!

===== Running Fold 2 =====


...

💾 Checkpoints for OOF and Test saved for Fold 3!

===== Running Fold 4 =====


/tmp/ipython-input-2876328625.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['brand_encoded'] = X_train['brand'].map(target_mean).fillna(y_train_fold.mean())
/tmp/ipython-input-2876328625.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val['brand_encoded'] = X_val['brand'].map(target_mean).fillna(y_train_fold.mean())


💾 Checkpoints for OOF and Test saved for Fold 4!

===== Running Fold 5 =====


/tmp/ipython-input-2876328625.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['brand_encoded'] = X_train['brand'].map(target_mean).fillna(y_train_fold.mean())
/tmp/ipython-input-2876328625.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val['brand_encoded'] = X_val['brand'].map(target_mean).fillna(y_train_fold.mean())


💾 Checkpoints for OOF and Test saved for Fold 5!


✅ Training complete! Assembling final submission file...
🎉 Success! New submission file created at:
/content/drive/My Drive/ML_Price_Prediction/submission_v2.csv

💾 Creating new, accurate OOF 'report card' file...
✅ New report card saved to:
/content/drive/My Drive/ML_Price_Prediction/dataset/oof_predictions_v2.csv


🏆 Your NEW model's validation SMAPE score is: 53.6517%


In [4]:
# ====================================================================
# 🔹 FINAL STEP: TRAINING AND SAVING DEPLOYMENT MODELS
# ====================================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os

print("🚀 Training and saving the final models for deployment...")

# --- Step 1: Load the final, clean training dataset ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
# New folder for the final, tuned models for deployment
DEPLOY_MODELS_PATH = os.path.join(MODELS_PATH, 'deployment_models/')
os.makedirs(DEPLOY_MODELS_PATH, exist_ok=True)

df_train = pd.read_csv(os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv'))

# --- Step 2: Prepare the full dataset for training ---
df_train['log_price'] = np.log1p(df_train['price'])
y = df_train['log_price']
features_to_drop = ['sample_id', 'catalog_content', 'image_link', 'price', 'log_price', 'item_name', 'unit']
features = [col for col in df_train.columns if col not in features_to_drop]
X = df_train[features]
print(f"Loaded full training data. X shape: {X.shape}")

# --- Step 3: Use your best parameters ---
# (Using the tuned parameters from your previous runs)
lgb_params = {
    'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 5000, # Using a high number, will use early stopping if available
    'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'num_leaves': 60, 'verbose': -1, 'n_jobs': -1, 'seed': 42
}
# If you have better params from Optuna, update them here:
# lgb_params.update(best_params_from_optuna)

# --- Step 4: Perform Target Encoding on the Full Dataset ---
target_mean = y.groupby(X['brand']).mean()
X['brand_encoded'] = X['brand'].map(target_mean).fillna(y.mean())
X_final = X.drop(columns=['brand']) # Final feature set

# --- Step 5: Define feature groups ---
structured_features = ['num_of_packs', 'net_qty', 'total_qty', 'is_qty_missing', 'brand_encoded']
image_features = [col for col in X_final.columns if col.startswith('img_embed_')]
text_features = [col for col in X_final.columns if col.startswith('text_embed_')]

# ====================================================================
# 🔹 Step 6: Train and SAVE each final model
# ====================================================================

# --- Train and Save Text Specialist Model ---
print("\n--- Training and saving Text Specialist model... ---")
text_cols = structured_features + text_features
final_model_text = lgb.LGBMRegressor(**lgb_params).fit(X_final[text_cols], y)
joblib.dump(final_model_text, os.path.join(DEPLOY_MODELS_PATH, 'final_model_text.pkl'))
print("✅ Text Specialist model saved.")

# --- Train and Save Vision Specialist Model ---
print("\n--- Training and saving Vision Specialist model... ---")
vision_cols = structured_features + image_features
final_model_vision = lgb.LGBMRegressor(**lgb_params).fit(X_final[vision_cols], y)
joblib.dump(final_model_vision, os.path.join(DEPLOY_MODELS_PATH, 'final_model_vision.pkl'))
print("✅ Vision Specialist model saved.")

# --- Train and Save Hybrid Model ---
print("\n--- Training and saving Hybrid model... ---")
hybrid_cols = structured_features + text_features + image_features
final_model_hybrid = lgb.LGBMRegressor(**lgb_params).fit(X_final[hybrid_cols], y)
joblib.dump(final_model_hybrid, os.path.join(DEPLOY_MODELS_PATH, 'final_model_hybrid.pkl'))
print("✅ Hybrid model saved.")

print(f"\n\n🎉 Success! All three final models are trained and saved in the '{DEPLOY_MODELS_PATH}' folder.")
print("You are now ready to use these files in your app.py!")

🚀 Training and saving the final models for deployment...
Loaded full training data. X shape: (75000, 965)


/tmp/ipython-input-3269726164.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['brand_encoded'] = X['brand'].map(target_mean).fillna(y.mean())



--- Training and saving Text Specialist model... ---
✅ Text Specialist model saved.

--- Training and saving Vision Specialist model... ---
✅ Vision Specialist model saved.

--- Training and saving Hybrid model... ---
✅ Hybrid model saved.


🎉 Success! All three final models are trained and saved in the '/content/drive/My Drive/ML_Price_Prediction/models/deployment_models/' folder.
You are now ready to use these files in your app.py!


In [5]:
# --- IN YOUR TRAINING NOTEBOOK, RUN THIS ONCE ---
import joblib
import os

MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
# Save the final list of columns that the models were trained on
joblib.dump(X_final.columns, os.path.join(MODELS_PATH, 'final_model_columns.pkl'))

print("✅ Final model column blueprint saved successfully!")

✅ Final model column blueprint saved successfully!


In [10]:
# ====================================================================
# 🔹 FINAL SCRIPT: INTERACTIVE PREDICTION IN COLAB
# ====================================================================

# --- Step 1: Install and Import ---
!pip install -q sentence-transformers torch torchvision joblib scikit-learn lightgbm
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
import torch
from PIL import Image
import joblib
import os
from google.colab import files # Import the file upload library
import io

print("🚀 Setting up the application and loading all model assets...")

# =============================
# 🔹 PART 1: LOAD ALL ASSETS
# =============================
# This happens only once when the script starts.

PROJECT_PATH = '/content/drive/My Drive/ML_Price_Prediction/'
MODELS_PATH = os.path.join(PROJECT_PATH, 'models/deployment_models/')
PREPROCESSING_PATH = os.path.join(PROJECT_PATH, 'models/')
BASE_PATH = os.path.join(PROJECT_PATH, 'dataset/')

device = "cuda" if torch.cuda.is_available() else "cpu"

model_text = joblib.load(os.path.join(MODELS_PATH, 'final_model_text.pkl'))
model_vision = joblib.load(os.path.join(MODELS_PATH, 'final_model_vision.pkl'))
model_hybrid = joblib.load(os.path.join(MODELS_PATH, 'final_model_hybrid.pkl'))
scaler = joblib.load(os.path.join(PREPROCESSING_PATH, 'scaler_v2.pkl'))
outlier_caps = np.load(os.path.join(PREPROCESSING_PATH, 'outlier_caps_v2.npy'), allow_pickle=True).item()

df_train = pd.read_csv(os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv'))
df_train['log_price'] = np.log1p(df_train['price'])
brand_target_map = df_train.groupby('brand')['log_price'].mean()
global_mean_price = df_train['log_price'].mean()

text_embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
img_model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
img_model.classifier = torch.nn.Identity()
img_model.to(device); img_model.eval()
preprocess = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

print("✅ All assets loaded successfully!")


# ====================================================================
# 🔹 PART 2: THE COMPLETE INFERENCE PIPELINE
# ====================================================================

def parse_product_details_v5(text):
    num_of_packs, net_qty_value = 1, 1.0
    text_lower = str(text).lower()
    pack_patterns = [r'\(pack of (\d+)\)', r'pack of (\d+)', r'(\d+)\s*per case', r'(\d+)\s*count', r'(\d+)\s*ct', r'(\d+)\s*pack', r'(\d+)-pack', r'(\d+)-ct', r'(\d+)\s*pcs']
    found_packs = [1]
    for pattern in pack_patterns:
        matches = re.findall(pattern, text_lower)
        if matches:
            for match in matches:
                found_packs.append(int(match))
    num_of_packs = max(found_packs)
    qty_pattern = r'(\d+\.?\d*)\s*-?\s*(fluid ounce|fl oz|ounces|ounce|oz|grams|gram|g|pounds|pound|lbs|lb|liters|liter|l|milliliters|milliliter|ml)\b'
    qty_match = re.search(qty_pattern, text_lower)
    if qty_match:
        net_qty_value = float(qty_match.group(1))
    return num_of_packs, net_qty_value

def get_embedding_from_image(pil_image, model, preprocess_func):
    try:
        img = pil_image.convert("RGB")
        batch_t = torch.unsqueeze(preprocess_func(img), 0).to(device)
        with torch.no_grad():
            embedding = model(batch_t)
        return embedding.cpu().numpy().flatten()
    except Exception:
        return np.zeros(576)

def predict_price(catalog_content, pil_image):
    print("\n--- Starting prediction pipeline for the new product ---")

    # 1. Parse Text & Create Engineered Features
    num_of_packs, net_qty = parse_product_details_v5(catalog_content)
    brand = catalog_content.split(' ')[0]
    total_qty = num_of_packs * net_qty
    is_qty_missing = 1 if num_of_packs == 1 and net_qty == 1.0 else 0
    brand_encoded = brand_target_map.get(brand, global_mean_price)
    print("✅ Text parsed and features engineered.")

    # 2. Generate Embeddings
    text_embedding = text_embedder.encode(catalog_content)
    image_embedding = get_embedding_from_image(pil_image, img_model, preprocess)
    print("✅ Image and text embeddings generated.")

    # 3. Assemble, Transform, and Align
    feature_dict = {
        'num_of_packs': num_of_packs, 'net_qty': net_qty, 'total_qty': total_qty,
        'is_qty_missing': is_qty_missing, 'brand_encoded': brand_encoded
    }
    for i, val in enumerate(image_embedding): feature_dict[f'img_embed_{i}'] = val
    for i, val in enumerate(text_embedding): feature_dict[f'text_embed_{i}'] = val
    inference_df = pd.DataFrame([feature_dict])

    for col, cap_value in outlier_caps.items():
        if col in inference_df.columns:
            inference_df.loc[inference_df[col] > cap_value, col] = cap_value

    cols_to_scale = ['num_of_packs', 'net_qty', 'total_qty']
    inference_df[cols_to_scale] = scaler.transform(inference_df[cols_to_scale])
    print("✅ Data transformed and aligned.")

    # 4. Predict and Format Output
    # This robust method ensures we only use the features the model was trained on
    text_cols = [f for f in model_text.feature_name_ if f in inference_df.columns]
    vision_cols = [f for f in model_vision.feature_name_ if f in inference_df.columns]
    hybrid_cols = [f for f in model_hybrid.feature_name_ if f in inference_df.columns]

    pred_text = model_text.predict(inference_df[text_cols])[0]
    pred_vision = model_vision.predict(inference_df[vision_cols])[0]
    pred_hybrid = model_hybrid.predict(inference_df[hybrid_cols])[0]

    ensemble_log_pred = (pred_text + pred_vision + pred_hybrid) / 3
    final_price = np.expm1(ensemble_log_pred)

    return f"${final_price:.2f}"

# ====================================================================
# 🔹 PART 3: INTERACTIVE INPUT AND PREDICTION
# ====================================================================

print("\n--- Please provide the product details below ---")

# --- 3.1 Get Text Input ---
print("1. Paste your catalog content below (and press Enter):")
new_catalog_content = input()

# --- 3.2 Get Image Input ---
print("\n2. Upload your product image:")
uploaded = files.upload()

# --- 3.3 Run Prediction ---
if uploaded:
    # Get the filename of the first uploaded file
    filename = next(iter(uploaded))
    print(f"\nProcessing uploaded file: {filename}")

    # Open the uploaded image file in memory
    pil_image = Image.open(io.BytesIO(uploaded[filename]))

    # Run the full prediction pipeline
    predicted_price = predict_price(new_catalog_content, pil_image)

    print("\n\n==============================================")
    print(f"🧠 Predicted Optimal Price: {predicted_price}")
    print("==============================================")
else:
    print("\nNo file was uploaded. Please run the cell again to try another prediction.")

🚀 Setting up the application and loading all model assets...
✅ All assets loaded successfully!

--- Please provide the product details below ---
1. Paste your catalog content below (and press Enter):
Pureheart Nutreat Salted Pistachios (1000 gm) Natural Premium Lightly Roasted Pista/Dry Fruit - Delicious & Crunchy - Reusable Jar

2. Upload your product image:


Saving Screenshot 2025-10-18 221747.png to Screenshot 2025-10-18 221747 (4).png

Processing uploaded file: Screenshot 2025-10-18 221747 (4).png

--- Starting prediction pipeline for the new product ---
✅ Text parsed and features engineered.
✅ Image and text embeddings generated.
✅ Data transformed and aligned.


🧠 Predicted Optimal Price: $15.40
